# Phase-Aware Learning Rate Restarts — Medium World

**Question:** Can LR restarts at genuine changes in the task distribution improve same-object Binding without excessive interference with earlier skills? Keep the existing `AttributeBindingViT`, fixed dataset, curriculum question streams, and 9,000-update budget. Compare Constant, Global Cosine, and Reset Cosine first; then Global and Reset ReduceLROnPlateau.

A **reset** never restores model parameters or AdamW moments. It only restarts the learning-rate policy. Consecutive blocks with the same task mix are merged (notably the two Pair-only blocks), and Fully Mixed has no reset boundary. The final mixed phase restarts at `5e-5` when it represents a genuine change. Treat this as an optimization-policy comparison, **not proof of an independent effect of scheduler history**; reset policies also change the cumulative applied LR.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEEDS = [42, 43, 44, 45, 46]
RUN_SEEDS = SEEDS
SCHEDULES = ("prereq_mixed_pair_focus", "staged_rehearsal", "scaffolded_spiral", "mixed")
RUN_SCHEDULES = SCHEDULES
POLICIES = ("constant", "global_cosine", "reset_cosine", "global_plateau", "reset_plateau")
RUN_POLICIES = POLICIES[3:]  # After Phase A, use POLICIES[3:] for the plateau study
BATCH_SIZE, EVAL_EVERY, TOTAL_STEPS = 50, 100, 9000
INITIAL_LR, FINAL_MIXED_LR, MIN_LR = 1e-4, 5e-5, 1e-6
FACTOR, PATIENCE, THRESHOLD = 2/3, 2, 0.001
WEIGHT_DECAY, CLIP_NORM = 0.0, 1.0
MODEL_DIM, PATCH_SIZE, MODEL_DEPTH = 48, 4, 1
AUGMENT, DEVICE = True, "cuda"
DATA_DIR = PROJECT_ROOT / "outputs/medium_world/dataset"
OUT_DIR = PROJECT_ROOT / "outputs/medium_world/experiments/phase_aware_lr"
RESULTS_FILE = OUT_DIR / "schedule_comparison.json"

In [2]:
import json, hashlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.scaffolded_spiral import make_spiral_blocks
from src.medium_visual_world.lr_policies import (
    LRPolicy, effective_phases, evaluation_steps
)
from src.medium_visual_world.prediction_tracking import PredictionTrace, evaluate_predictions

device = torch.device(DEVICE)
torch.set_num_threads(min(4, torch.get_num_threads()))
train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
ood_dev = MediumDataset(DATA_DIR / "ood_dev.jsonl")
hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
          for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl")}
print("Device:", device, "| Questions:", len(train_data), "| Policies:", RUN_POLICIES)

Device: cuda | Questions: 10000 | Policies: ('global_plateau', 'reset_plateau')


## Audit real transitions and exposure

Only **changes in task mixture** trigger resets: Prerequisite-Mixed at steps 5,400 and 7,200; Staged Rehearsal at 3,000, 4,950, and 7,200; Scaffolded Spiral at 2,800, 4,450, 6,100, and 7,200; Fully Mixed never. Each question must appear **45 times** in every run. The `audit_lr_reset.py` script checks all five seeds, exact exposure, historical schedule agreement, and scheduler behavior independently.

In [3]:
plans = []
for schedule in SCHEDULES:
    blocks, plan = make_spiral_blocks(train_data.rows, schedule, SEEDS[0])
    segments = effective_phases(plan["phases"])
    plans.extend({"schedule": schedule, "phase": "+".join(seg["names"]),
                  "start": seg["start_step"], "end": seg["end_step"],
                  "binding_share": seg["mix"][-1]} for seg in segments)
    assert sum(len(ids) for _, ids in blocks) == TOTAL_STEPS * BATCH_SIZE
    assert len(Counter(i for _, ids in blocks for i in ids)) == len(train_data)
    assert set(Counter(i for _, ids in blocks for i in ids).values()) == {45}
display(pd.DataFrame(plans))

,schedule,phase,start,end,binding_share
0,prereq_mixed_pair_focus,prereq_mixed,0,5400,0.000000
1,prereq_mixed_pair_focus,pair_early+pair_focus,5400,7200,1.000000
2,prereq_mixed_pair_focus,common_mixed,7200,9000,0.250000
3,staged_rehearsal,attributes,0,3000,0.000000
4,staged_rehearsal,conjunction,3000,4950,0.000000
5,staged_rehearsal,binding,4950,7200,0.800000
6,staged_rehearsal,common_mixed,7200,9000,0.250000
7,scaffolded_spiral,attributes,0,2800,0.000000
8,scaffolded_spiral,conjunction_rehearsal,2800,4450,0.000000
9,scaffolded_spiral,binding_specialization,4450,6100,0.787879


## Train and save real observations

For each seed and curriculum, all LR policies start from the same seeded initialization and training question order. We evaluate the same ID/OOD questions throughout. Every 100 global updates plus true transition checkpoints and local plateau-monitor checkpoints are logged. AULC always uses the shared 100-update grid to avoid evaluation-frequency differences.

The notebook resumes after **completed** runs; only the current run would need to be restarted after an interruption. Do not switch model architecture or choose the best dev checkpoint during this experiment.

In [4]:
def binding_auc(history):
    points = [(h["step"], h["id_dev"]["by_template"]["pair"])
              for h in history if h["step"] % EVAL_EVERY == 0]
    return sum((b[0]-a[0])*(a[1]+b[1])/2
               for a,b in zip(points[:-1],points[1:])) / TOTAL_STEPS

def run_one(schedule, policy, seed):
    set_seed(seed)
    model = AttributeBindingViT(dim=MODEL_DIM, patch_size=PATCH_SIZE,
                                depth=MODEL_DEPTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR,
                                   weight_decay=WEIGHT_DECAY)
    blocks, plan = make_spiral_blocks(train_data.rows, schedule, seed)
    segments = effective_phases(plan["phases"])
    controller = LRPolicy(policy, optimizer, segments, initial_lr=INITIAL_LR,
        final_mixed_lr=FINAL_MIXED_LR, min_lr=MIN_LR, factor=FACTOR,
        patience=PATIENCE, threshold=THRESHOLD, check_every=EVAL_EVERY)
    eval_steps = set(evaluation_steps(plan["phases"], segments, EVAL_EVERY))
    transitions = {part["start_step"] for part in segments[1:]}
    trace = PredictionTrace({"id_dev": id_dev, "ood_dev": ood_dev},
        metadata={"seed": seed, "schedule": schedule, "policy": policy,
                  "dataset_hashes": hashes})
    history, seen = [], Counter()
    lr_by_task = Counter()
    step, running_loss, running_norm, batches = 0, 0.0, 0.0, 0

    def evaluate_step(phase, used_lr=None):
        nonlocal running_loss, running_norm, batches
        scores = {}
        for split, dataset in (("id_dev", id_dev), ("ood_dev", ood_dev)):
            summary, predictions = evaluate_predictions(model, dataset, device, BATCH_SIZE)
            scores[split] = summary
            trace.add(split, step, predictions)
        controller.observe(step, scores["id_dev"]["loss"])
        history.append({"step": step, "phase": phase,
            "lr_last_update": used_lr,
            "lr_next_update": controller.preview(step) if step<TOTAL_STEPS else None,
            "batch_train_loss": running_loss/batches if batches else None,
            "mean_gradient_norm_preclip": running_norm/batches if batches else None,
            "cumulative_task_presentations": dict(seen),
            "cumulative_lr_by_task": dict(lr_by_task),
            "real_transition": step in transitions, **scores})
        running_loss, running_norm, batches = 0.0, 0.0, 0
        print(f"{schedule:23} {policy:16} seed={seed} step={step:4} "
              f"ID Pair={scores['id_dev']['by_template']['pair']:.3f} "
              f"OOD Pair={scores['ood_dev']['by_template']['pair']:.3f}", flush=True)

    evaluate_step("initial")
    for phase, indices in blocks:
        for start in range(0, len(indices), BATCH_SIZE):
            sample_ids = indices[start:start+BATCH_SIZE]
            x, q, y = train_data.batch(sample_ids, device)
            if AUGMENT:
                x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            used_lr = controller.before_update(step)
            loss = nn.functional.cross_entropy(model(x, q), y)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
            step += 1
            running_loss += loss.item()
            running_norm += float(grad_norm)
            batches += 1
            tasks = Counter(train_data.tasks[i] for i in sample_ids)
            seen.update(tasks)
            lr_by_task.update({task: count*used_lr for task,count in tasks.items()})
            if step in eval_steps:
                evaluate_step(phase, used_lr)
    trace_name = f"prediction_traces/{schedule}_{policy}_seed_{seed}.npz"
    trace.save(OUT_DIR / trace_name)
    assert step == TOTAL_STEPS
    assert all(seen[task] == 112500 for task in TASKS)
    return {"seed": seed, "schedule": schedule, "policy": policy,
            "status": "completed", "steps_run": step, "phase_plan": plan,
            "effective_phases": segments, "reset_events": controller.reset_events,
            "history": history, "final": history[-1],
            "id_pair_aulc": binding_auc(history),
            "cumulative_lr_by_task": dict(lr_by_task),
            "prediction_trace": trace_name}

In [5]:
CONFIG = {"schedules": list(SCHEDULES), "policies": list(POLICIES),
          "seeds": SEEDS, "batch_size": BATCH_SIZE, "eval_every": EVAL_EVERY,
          "total_steps": TOTAL_STEPS, "initial_lr": INITIAL_LR,
          "final_mixed_lr": FINAL_MIXED_LR, "min_lr": MIN_LR,
          "factor": FACTOR, "patience": PATIENCE, "threshold": THRESHOLD,
          "weight_decay": WEIGHT_DECAY, "clip_norm": CLIP_NORM,
          "dim": MODEL_DIM, "patch_size": PATCH_SIZE, "depth": MODEL_DEPTH,
          "augmentation": AUGMENT, "dataset_hashes": hashes}

def save_report(report):
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    temp = RESULTS_FILE.with_suffix(".json.tmp")
    temp.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    temp.replace(RESULTS_FILE)

report = (json.loads(RESULTS_FILE.read_text(encoding="utf-8")) if RESULTS_FILE.exists()
          else {"config": CONFIG, "environment": environment(), "runs": []})
assert report["config"] == CONFIG, "Existing report belongs to another configuration"
completed = {(r["seed"],r["schedule"],r["policy"]) for r in report["runs"]}
for seed in RUN_SEEDS:
    for schedule in RUN_SCHEDULES:
        for policy in RUN_POLICIES:
            key = (seed,schedule,policy)
            if key in completed:
                print("Already completed:",key)
                continue
            result = run_one(schedule,policy,seed)
            report["runs"].append(result)
            save_report(report)
            print("Saved:",key)
print("Saved runs:",len(report["runs"]),"of",len(SEEDS)*len(SCHEDULES)*len(POLICIES))

prereq_mixed_pair_focus global_plateau   seed=42 step=   0 ID Pair=0.500 OOD Pair=0.500
prereq_mixed_pair_focus global_plateau   seed=42 step= 100 ID Pair=0.492 OOD Pair=0.500
prereq_mixed_pair_focus global_plateau   seed=42 step= 200 ID Pair=0.508 OOD Pair=0.500
prereq_mixed_pair_focus global_plateau   seed=42 step= 300 ID Pair=0.512 OOD Pair=0.500
prereq_mixed_pair_focus global_plateau   seed=42 step= 400 ID Pair=0.508 OOD Pair=0.480
prereq_mixed_pair_focus global_plateau   seed=42 step= 500 ID Pair=0.508 OOD Pair=0.482
prereq_mixed_pair_focus global_plateau   seed=42 step= 600 ID Pair=0.504 OOD Pair=0.487
prereq_mixed_pair_focus global_plateau   seed=42 step= 700 ID Pair=0.504 OOD Pair=0.485
prereq_mixed_pair_focus global_plateau   seed=42 step= 800 ID Pair=0.500 OOD Pair=0.482
prereq_mixed_pair_focus global_plateau   seed=42 step= 900 ID Pair=0.500 OOD Pair=0.482
prereq_mixed_pair_focus global_plateau   seed=42 step=1000 ID Pair=0.500 OOD Pair=0.482
prereq_mixed_pair_focus global_p

## Inspect the executed run inventory

Do not infer final conclusions here. The analysis notebook requires a complete five-seed grid for its selected LR policies. To add the Plateau policies later, change only `RUN_POLICIES = POLICIES[3:]` above and rerun this training notebook.

In [6]:
summary = pd.DataFrame([{"seed": r["seed"], "schedule": r["schedule"],
    "policy": r["policy"], "updates": r["steps_run"],
    "checks": len(r["history"]), "restarts": len(r["reset_events"]),
    "ID_Pair": r["final"]["id_dev"]["by_template"]["pair"],
    "OOD_Pair": r["final"]["ood_dev"]["by_template"]["pair"]}
    for r in report["runs"]])
display(summary.sort_values(["seed","schedule","policy"]))

,seed,schedule,policy,updates,checks,restarts,ID_Pair,OOD_Pair
9,42,mixed,constant,9000,91,0,0.656250,0.6125
10,42,mixed,global_cosine,9000,91,0,0.589844,0.5475
66,42,mixed,global_plateau,9000,91,0,0.597656,0.5500
11,42,mixed,reset_cosine,9000,91,0,0.589844,0.5475
67,42,mixed,reset_plateau,9000,91,0,0.578125,0.5625
...,...,...,...,...,...,...,...,...
51,46,staged_rehearsal,constant,9000,114,0,0.621094,0.5950
52,46,staged_rehearsal,global_cosine,9000,114,0,0.550781,0.5225
94,46,staged_rehearsal,global_plateau,9000,114,0,0.527344,0.5075
53,46,staged_rehearsal,reset_cosine,9000,114,3,0.574219,0.5500
